In [1]:
import os
import numpy as np
import cupy as cp
from numba import cuda, float64
from math import exp, pi, atan
from time import time

Initialization of the GPU block sizes 

In [3]:
N_LX: int = 32 
N_LY: int = 32 
N_SPINS: int = N_LX * N_LY 
N2_SPINS: int = 2 * N_SPINS

In [4]:
N_SUB: int = N2_SPINS // 2 
I_BEGIN: int = (N2_SPINS - N_SUB) // 2 
I_END: int = I_BEGIN + N_SUB

In [5]:
TpB: int = 256 
NBlock: int = (N_SPINS + TpB - 1) // TpB 

Loading the file of parameters.

In [6]:
def load_paras(file_name):
    with open(file_name,'r') as f:
        return np.float32(f.readlines()[2].split('\t'))

Loading the file of neighbors.

In [7]:
def load_nbs(filename):  
    with open(filename,'r') as file:
        return np.array([line.split('|')[1].split() for line in file.readlines()],dtype=np.int32)

In [ ]:
Loading the file of spin textures.

In [8]:
def load_spins(filename):  
    with open(filename,'r') as file:
        return cp.array([line.split()[3:6] for line in file.readlines()],dtype=cp.float64)

Fermi districtuion function

In [9]:
def fermi(e,t):
    return cp.array([1.0/(exp((e[i]-MU)/t)+1) for i in range(0, len(e))])

In [ ]:
Calculation of the spin chirality and topp

In [10]:
def get_one_chi(v1,v2,v3):
    return cp.linalg.det(cp.array([v1,v2,v3],dtype=np.float64))  #计算出给定方阵的行列式

In [11]:
def get_one_tc(v1,v2,v3):
    x = 1 + cp.dot(v1,v2) + cp.dot(v2,v3) + cp.dot(v3,v1)  # cp.dot()获取两个元素的乘积，如果是两个一维数组则获取内积，如果是两个多维数组则为矩阵乘法
    y = get_one_chi(v1,v2,v3)
    if x < 0:
        return 0.5 + atan(y/x)/2/pi if y>0 else -0.5 + atan(y/x)/2/pi
    elif x > 0:
        return atan(y/x)/2/pi
    else:
        return 0.25 if y>0 else -0.25

In [12]:
 def chi_tc_sum(sp,nbs):
    chi0,chi1,chi2,chi3 = cp.zeros(len(sp)),cp.zeros(len(sp)),cp.zeros(len(sp)),cp.zeros(len(sp))
    tc0,tc1,tc2,tc3 = cp.zeros(len(sp)),cp.zeros(len(sp)),cp.zeros(len(sp)),cp.zeros(len(sp))
    for i in range(0, N_SPINS):
        chi0[i] = get_one_chi(sp[i],sp[nbs[i,0]],sp[nbs[i,3]])
        chi1[i] = get_one_chi(sp[i],sp[nbs[i,1]],sp[nbs[i,2]])
        chi2[i] = get_one_chi(sp[i],sp[nbs[i,2]],sp[nbs[i,0]])
        chi3[i] = get_one_chi(sp[i],sp[nbs[i,3]],sp[nbs[i,1]])
        tc0[i] = get_one_tc(sp[i],sp[nbs[i,0]],sp[nbs[i,3]])
        tc1[i] = get_one_tc(sp[i],sp[nbs[i,1]],sp[nbs[i,2]])
        tc2[i] = get_one_tc(sp[i],sp[nbs[i,2]],sp[nbs[i,0]])
        tc3[i] = get_one_tc(sp[i],sp[nbs[i,3]],sp[nbs[i,1]])
    return (cp.sum(chi0)+cp.sum(chi1)+cp.sum(chi2)+cp.sum(chi3))/16/pi,\
           (cp.sum(tc0)+cp.sum(tc1)+cp.sum(tc2)+cp.sum(tc3))/4

##### For further
The summations of tc and chilarity have not been improved very well. \
tc和手性的总结没有得到很好的改进

In [13]:
@cuda.jit 
def get_hamiltonian_cuda(ham, nbs, sp): 
    i = cuda.grid(1) 
    if i < N_SPINS: 
        ham[i, nbs[i, 0]] = HOP
        ham[i, nbs[i, 1]] = HOP
        ham[i, nbs[i, 2]] = HOP
        ham[i, nbs[i, 3]] = HOP
        ham[i + N_SPINS, nbs[i, 0] + N_SPINS] = HOP
        ham[i + N_SPINS, nbs[i, 1] + N_SPINS] = HOP
        ham[i + N_SPINS, nbs[i, 2] + N_SPINS] = HOP 
        ham[i + N_SPINS, nbs[i, 3] + N_SPINS] = HOP 
    cuda.syncthreads() 
    if i < N_SPINS: 
        ham[i, i] += J_H*sp[i, 2] 
        ham[i + N_SPINS, i + N_SPINS] += -J_H*sp[i, 2] 
        ham[i, i + N_SPINS] += J_H*complex(sp[i, 0], -sp[i, 1]) 
        ham[i + N_SPINS, i] += J_H*complex(sp[i, 0], sp[i, 1])
    cuda.syncthreads()

##### A little difference between this version and Jie-Xiang's.
I put another cuda.syncthreads() on the bottom of the part by Barry's suggestion. (Don't ask me who is Barry.)

In [14]:
def uxr(us): 
    """ 
    neighbor of us along x 
    :param us: the bundle of eigenvectors we need   
    :return: 
    """ 
    roll = cp.zeros([N_SUB, N2_SPINS], dtype=np.complex128) 
    roll[:,0:N_SPINS] = cp.roll(us[:,0:N_SPINS], 1, axis = 1) 
    roll[:,N_SPINS:] = cp.roll(us[:,N_SPINS:], 1, axis = 1) 
    roll[:,0::N_LX] = us[:,N_LX-1::N_LX]
    return roll

In [15]:
def uyr(us): 
    """ 
    neighbor of us along y 
    :param us: the bundle of eigenvectors we need  
    :return: 
    """ 
    roll = cp.zeros([N_SUB, N2_SPINS], dtype=np.complex128) 
    roll[:, 0: N_SPINS] = cp.roll(us[:, 0:N_SPINS], N_LX, axis = 1) 
    roll[:, N_SPINS: ] = cp.roll(us[:, N_SPINS: ], N_LX, axis =1) 
    return roll

#####  Current operators work on the eigenvectors
I stiil use 'roll' to achieve the eigenvecotors after current operators woked on them. Here, I roll all the vectors together. BTW, The highlighted line in uxr in Jie-Xiang rewrited version has a typo. \
##### 当前操作符处理特征向量 
我仍然使用“roll”来实现当前操作符对其进行操作后的特征向量。

In [16]:
def get_sigma(e,f,u,ux,uy): 
    sxx, sxy = 0.0, 0.0
    i = 1
    while i < N_SUB:
        e_diff = e - cp.roll(e,i)
        f_diff = f - cp.roll(f,i)
        fac = f_diff/(ETA*ETA+e_diff*e_diff)
        jx = cp.sum(u.conjugate()*cp.roll(ux,i,axis=0),axis = 1)#- ux*cp.roll(u,i,axis=0).conjugate(),axis=1)
        jy = cp.sum(u.conjugate()*cp.roll(uy,i,axis=0),axis = 1)# - uy*cp.roll(u,i,axis=0).conjugate(),axis=1)
        sxx += cp.sum(fac * (jx.conjugate()*jx/ (-e_diff + ETA * 1j)).real)*ETA
        sxy += cp.sum(fac * (jx * jy.conjugate()).imag)
        i += 1
    return sxx, sxy

##### For conductivy calculation:
Every step above works on all the (eigen)values and (eigen)vectors by using 'roll'. One Loop is enough.
##### 对于电导率计算：
上面的每一步都通过使用“roll”来处理所有（本征）值和（本征）向量。一个循环就足够了。

#####  Some declarations:
The cupy package is very proper to do the calculation with tons of vectors and  matrices. Now, the cupy can realize almost all the functions of numpy with GPU acceleration., much much better than one year before. Using Numba to parrellel and accelerate the calculation has lots of problems. Especially three problems makes this project more complicated. 

1. Lots of functions in numpy are not supported in Numba.

2. The complicated computation which I mean the ones beyond linear is not as fast as I anticipate based because of the design of GPU. In getting Hamiltonian, it is so proper to use Numba to parrallel the process. 

3. The GPU memory limitation. Unproper planning the blocks and threads will make the program disaster. 

#####  一些声明：
cupy软件包非常适合用大量向量和矩阵进行计算。现在，cupy可以通过GPU加速实现numpy的几乎所有功能。，比一年前好多了。使用Numba进行parrellel和加速计算有很多问题。尤其是三个问题使这个项目更加复杂。

1. Numba不支持numpy中的许多函数。

2. 由于GPU的设计，复杂的计算（我指的是超出线性的计算）没有预期的那么快。在得到哈密顿量时，使用Numba来描述这个过程是非常恰当的。

3. GPU内存限制。不正确地规划块和线程将导致程序灾难。

##### Example. 
I will use a 32*32 lattice to demonstrate the process. timing is also used to compare the GPU and CPU part.

In [17]:
# try:
#     HOP, J_H, MU, ETA = load_paras('paras.txt')
#     print('Parameters have been loaded!')
#     #print(HOP)
#     #print(J_H)
# except:
#     print('Load failed!')
#     pass

In [18]:
# HOP, J_H, MU, ETA

In [19]:
# #path = "out_2dsquare_snap_32_d030_b006_2w1"
# spins = 'T1.5000_spins_0.txt'  
# neighbors = 'neighbors.txt'    
# t = 1.5

In [20]:
# sp = load_spins(spins)
# nbs = load_nbs(neighbors)

##### The Hamiltonian

In [22]:
ham = cp.zeros([N2_SPINS, N2_SPINS], dtype=cp.complex128)# for cupy
ham1 = np.zeros([N2_SPINS, N2_SPINS], dtype=np.complex128)# for numpy

In [23]:
get_hamiltonian_cuda[NBlock, TpB](ham1, nbs, sp)
get_hamiltonian_cuda[NBlock, TpB](ham, nbs, sp)

E:\ANACONDA\envs\env1\lib\site-packages\numba\cuda\compiler.py:726: NumbaPerformanceWarning: Grid size (4) < 2 * SM count (32) will likely result in GPU under utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


NameError: name 'nbs' is not defined

#### Diagnolize the Hamiltonian

In [24]:
begin = time()
e_1,u_1 = np.linalg.eigh(ham1)
u_1 = u_1.transpose()
print(time()-begin) #CPU, all numpy

3.689668893814087


In [25]:
begin = time()
e_, u_ = cp.linalg.eigh(ham)
u_ = u_.transpose() # I use transpose to make the column vectors change to the row vectors.
# it is more convenient than Jie-Xiang's express.
print(time()-begin) #GPU, all cupy

5.177665710449219


In [26]:
e_1[0]-e_[0]

array(0.)

In [27]:
begin = time()
e_2,u_2 = np.linalg.eigh(ham)
u_2 = u_2.transpose()
print(time()-begin) #numpy diagnolize the ham, using cupy ham
#cupy cannot diagnolize the numpy matrix, it will give an error.

1.022263765335083


In [28]:
e_2[0]-e_[0]

array(0.)

In [29]:
e_[0],e_[1023], e_[1024],e_[2047], MU, t

NameError: name 'MU' is not defined

In [44]:
e = e_[I_BEGIN: I_END]
u = u_[I_BEGIN: I_END]
f = fermi(e, 1.5)
ux = uxr(u)
uy = uyr(u)

In [45]:
len(e), len(u), len(f), len(ux),len(u[0])

(1024, 1024, 1024, 1024, 2048)

In [46]:
begin = time()
sxx, sxy = get_sigma(e,f,u,ux,uy)
time()-begin

2.391414165496826

In [50]:
sxx, sxy

(array(0.65431239), array(0.05865553))

In [30]:
if __name__ == "__main__": 
    HOP, J_H, MU, ETA = load_paras('paras.txt')
    nbs = load_nbs(neighbors)
    
    begin = time()
    
    sp = load_spins(spins)
    ham = cp.zeros([N2_SPINS, N2_SPINS], dtype=cp.complex128)
    get_hamiltonian_cuda[NBlock, TpB](ham, nbs, sp)
    e_, u_ = cp.linalg.eigh(ham) # 返回复厄米矩阵（共轭对称）或实对称矩阵的特征值和特征向量。
    u_ = u_.transpose()  # 转置
    
    e = e_[I_BEGIN: I_END]  # [512:1536]
    u = u_[I_BEGIN: I_END]
    
    f = fermi(e, t)
    ux = uxr(u)
    uy = uyr(u)
    
    sxx, sxy = get_sigma(e,f,u,ux,uy)
    print("sigma_xx = {0} \t sigma_xy = {1}".format(sxx, sxy))
    print("time used: {0}".format(time()-begin))

FileNotFoundError: [Errno 2] No such file or directory: 'paras.txt'

##### The conditions:
1. The NVIDIA GPU in my computer is 1660Ti.
2. Using Jupyter Enviroment.

##### Some declarations:
1. The energy cut-off can be selected more precisely in further calculation,for example, using adabtic conditions in strong coupling region.

2. Comparing to use Numba, cupy can reduce the time of communication between host(CPU+memory) and device(GPU). Also, it can reduce the frequency for us to transfer the data between host and device by hand.

3. I prefer to manipulate the vectors by group. With the advantages of cupy, it is easy to accelerate the calculation by GPU through cupy.

4. In the scale of data in this project. Many steps of calculation have no time difference between cupy and numpy. (The diagnolization has a very large difference in time. I have tested very large data. The cupy is slower than numpy, probably because of the XX60 graphic card, the GPU memeory is not large enough.)

##### 一些声明：

1. 在进一步的计算中可以更精确地选择能量截止，例如，使用强耦合区域中的自适应条件。

2. 与使用Numba相比，cupy可以减少主机（CPU+内存）和设备（GPU）之间的通信时间。此外，它可以减少我们手动在主机和设备之间传输数据的频率。

3. 我更喜欢按组操纵向量。由于cupy的优点，很容易通过cupy加速GPU的计算。

4. 在本项目的数据规模中。许多计算步骤在cupy和numpy之间没有时间差。（诊断在时间上有很大的差异。我测试了非常大的数据。cupy比numpy慢，可能是因为XX60显卡，GPU内存不够大。）